In [ ]:
# Empty code cell required for "ruff" extension

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.config import data_path
from src.spark_utils import get_spark
import pyspark.sql.functions as F
from src.datasets import read_qqp, read_qqp_benchmark, read_feature_benchmark
from src.metrics import ModelMetrics, extract_metrics
from src.fingerprint import FingerprintParams, create_fingerprints

%load_ext autoreload
%autoreload 2

spark = get_spark()

In [ ]:
from pyspark.sql.types import StructField, StructType, BinaryType, IntegerType

from src.near_duplicates import detect_near_duplicate_v2

schema = StructType([
    StructField("ID", IntegerType(), nullable=False),
    StructField("fingerprint", BinaryType(), nullable=False)
])

data1 = [
    (1, bytes.fromhex("0123456789ABCDEF")),
    (2, bytes.fromhex("00000000FFFFFFFF")),
    (3, bytes.fromhex("FFFFFFFF00000000"))
]

# Dati per il secondo DataFrame
data2 = [
    (10, bytes.fromhex("00000111FFFFFFFF")),
    (20, bytes.fromhex("8877000044332211")),
    (30, bytes.fromhex("CAFEBABE00112233"))
]

df1 = spark.createDataFrame(data=data1, schema=schema)
df2 = spark.createDataFrame(data=data2, schema=schema)

detect_near_duplicate_v2(df1, df2, 64, 3, "ID", "fingerprint").show()

In [ ]:
from src.near_duplicates import detect_near_duplicates_pairs
df = read_qqp().limit(40)
df = create_fingerprints(df, "embeddings", "data1", "emb1", 64, None)
df = create_fingerprints(df, "embeddings", "data2", "emb2", 64, None)
df = detect_near_duplicates_pairs(df, "emb1", "emb2", 6)
# 3, 5, 4, 1, 4, 0, 2, 3, 3, 5
df.show(n=40, truncate=False)

In [ ]:
spark.stop()